### Import the Data

In [ ]:
%run  Data_preparation_TCGA.ipynb

### Import the Model

In [ ]:
from Classification_model import *

### Training Process

In [ ]:
train_loader = DataLoader(training_set, batch_size=1024, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=5096, shuffle=False)

In [ ]:
torch.manual_seed(0)

num_hiddens_genotype = 16
num_hiddens_final = 16

model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx))

In [ ]:
def create_term_mask(term_direct_gene_map, gene_dim, device):

    term_mask_map = {}

    for term, gene_set in term_direct_gene_map.items():

        mask = torch.zeros(len(gene_set), gene_dim)

        for i, gene_id in enumerate(gene_set):
            mask[i, gene_id] = 1

        mask_gpu = torch.autograd.Variable(mask)

        term_mask_map[term] = mask_gpu.to(device)

    return term_mask_map

term_mask_map = create_term_mask(model.term_direct_gene_map, num_genes, device = DEVICE)


In [ ]:
model_loaded = torch.load('model_032_updated.pt', map_location='cpu')

state_dict = model_loaded.state_dict()
current_state_dict = model.state_dict()

filtered_state_dict = {
    k: v for k, v in state_dict.items()
    if k in current_state_dict and v.size() == current_state_dict[k].size()
}

model.load_state_dict(filtered_state_dict, strict=False)


In [ ]:
model.to(DEVICE)
learning_rate = 0.003
torch.manual_seed(0)
loss_list = []
accu_list = []
train_epochs = 500

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), eps=1e-05, weight_decay=1e-4)


term_mask_map = create_term_mask(model.term_direct_gene_map, gene_dim=num_genes, device=DEVICE)

optimizer.zero_grad()

best_epoch = 0
best_accu = 0
best_model_path = "model_classification_set_teacher.pt"

for name, param in model.named_parameters():
    term_name = name.split('_')[0]

    if '_direct_gene_layer.weight' in name:
        param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 1
    else:
        param.data = param.data * 1

tepoch = tqdm.tqdm(range(train_epochs))
for epoch in tepoch:

    # Train
    model.train()
    train_predict = torch.zeros(0, 0).to(DEVICE)

    for i, (data, labels) in enumerate(train_loader):
        # Get the information from teacher model
        # Convert torch tensor to Variable

        # Forward + Backward + Optimize
        optimizer.zero_grad()  # zero the gradient buffer

        # Here term_NN_out_map is a dictionary
        logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(data.to(DEVICE))
        
        student_feats = term_NN_out_map

        if train_predict.size()[0] == 0:
            train_predict = aux_out_map["final"].data
        else:
            train_predict = torch.cat([train_predict, aux_out_map["final"].data], dim=0)

        loss_vae, class_loss, KLD = model.loss_log_vae(
            logits=logits, y=labels.to(DEVICE), mu=mu, log_var=log_var, beta=0.001
        )

        loss_intermidiate = model.intermediate_loss_cancer(aux_cancer_map, labels.to(DEVICE))


        total_loss = torch.mean(loss_vae + 0.2 * loss_intermidiate)
    
        
        tmp_loss = total_loss.item()
        
        total_loss.backward()

        for name, param in model.named_parameters():
            if "_direct_gene_layer.weight" not in name:
                continue
            term_name = name.split("_")[0]
            # print name, param.grad.data.size(), term_mask_map[term_name].size()
            if param.requires_grad and param.grad is not None:
                term_name = name.split("_")[0]
                param.grad.data = torch.mul(param.grad.data, term_mask_map[term_name])

        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        (inputdata, labels) = next(iter(test_loader))
        inputdata = inputdata.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(inputdata)
        preds = torch.argmax(logits, dim=1)
        accu = (preds == labels).float().mean().item()
    
        loss_list.append(tmp_loss)
        accu_list.append(accu)
    # if epoch % 10 == 0:
    if accu > best_accu:
        best_epoch = epoch
        best_accu = accu
        # torch.save(model, best_model_path)
    tepoch.set_postfix({"Epoch": epoch, "Loss": tmp_loss, "Accuracy": accu})
        
print(f"Epoch {best_epoch}: New best model saved with accuracy {best_accu:.4f}")
print("Training complete. Best model saved at:", best_model_path)


In [ ]:
with open('tcga_loss_list_set_para.txt', 'w') as f:
    for loss in loss_list:
        f.write(f"{loss}\n")

with open('tcga_accuracy_list_set_para.txt', 'w') as f:
    for accuracy in accu_list:
        f.write(f"{accuracy}\n")


In [ ]:
plt.plot(loss_list)

In [ ]:
plt.plot(accu_list)